## Download Text8 Model

In [ ]:
!wget -O dfm.pt "https://www.dropbox.com/scl/fi/rno9fq8mpjs2bdctz7o53/dfm.pt?rlkey=1ge1wxv14b4a46b730hbltwkg&dl=1"

## Load the Pre-Trained Model

In [1]:
import sys
sys.path.append('../')

import torch
import torch.nn.functional as F
from flow_model import GPT, GPTConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Text8 uses 27 characters (a-z and space). The mask token acts as the 28th token.
vocab_size = 28  
mask_token_id = 27

# Load the checkpoint
ckpt_path = 'dfm.pt'
checkpoint = torch.load(ckpt_path, map_location=device)

# Configure and instantiate the model
model_args = checkpoint['model_args']
model_args['vocab_size'] = vocab_size
gptconf = GPTConfig(**model_args)
model = GPT(gptconf)

# Strip out the DDP prefix ('_orig_mod.') if it exists
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k, v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

model.load_state_dict(state_dict)
model.eval()
model.to(device)
print("Model successfully loaded!")

# Create a decoder mapping (0 = space, 1-26 = a-z, 27 = mask)
itos = {0: ' '}
for i in range(1, 27):
    itos[i] = chr(ord('a') + i - 1)
itos[mask_token_id] = 'X'

def decode(token_list):
    return ''.join([itos.get(int(idx), '?') for idx in token_list])

Using device: cuda
GPT config GPTConfig(block_size=256, vocab_size=28, n_layer=12, n_head=12, n_embd=768, dropout=0.0, bias=False, qk_layernorm=True, do_x1_sc=True, mask_token_id=27, proper_timestep_emb=False, d3pm_loss_weighting=False, d3pm_loss_weighting_maxT=1000)
number of parameters: 86.16M
Model successfully loaded!


## Define Sampling Discrete Flow Model

In [2]:
def sample_discrete_flow(model, batch_size=4, block_size=256, dt=0.001, max_t=0.98, noise=0.0, x1_temp=1.0):
    """
    Minimal discrete diffusion flow sampling, updated to support self-conditioning (do_x1_sc).
    """
    samples = mask_token_id * torch.ones((batch_size, block_size), device=device, dtype=torch.long)
    mask_one_hot = torch.zeros((vocab_size,), device=device)
    mask_one_hot[mask_token_id] = 1.0
    
    # 1. Initialize the self-conditioning variable (x1_sc) as pure masks
    if model.config.do_x1_sc:
        x1_sc = mask_token_id * torch.ones_like(samples)
    
    t = 0.0
    with torch.no_grad():
        while t <= max_t:
            # 2. Forward pass: Use _run_net if self-conditioning, otherwise standard forward
            if model.config.do_x1_sc:
                logits = model._run_net(samples, t * torch.ones((batch_size,), device=device), x1=x1_sc)
            else:
                logits, _ = model(samples, t * torch.ones((batch_size,), device=device))
            
            # Get probabilities of the clean data
            pt_x1_probs = F.softmax(logits / x1_temp, dim=-1)
            
            # 3. Update the self-conditioning guess for the *next* iteration
            if model.config.do_x1_sc:
                x1_sc = torch.multinomial(pt_x1_probs.view(-1, vocab_size), num_samples=1).view(batch_size, block_size).long()
            
            # Identify which tokens are currently masked
            sample_is_mask = (samples == mask_token_id).view(batch_size, block_size, 1).float()
            
            # Probability of transitioning from a mask to a standard token
            step_probs = dt * pt_x1_probs * ((1 + noise * t) / (1 - t)) 
            step_probs = step_probs * sample_is_mask
            
            # Probability of transitioning from a standard token back to a mask (if noise > 0)
            step_probs += dt * (1 - sample_is_mask) * mask_one_hot.view(1, 1, -1) * noise
            step_probs = torch.clamp(step_probs, min=0.0, max=1.0)
            
            # Calculate the probability of staying exactly the same
            b_idx = torch.arange(batch_size, device=device).repeat_interleave(block_size)
            d_idx = torch.arange(block_size, device=device).repeat(batch_size)
            
            step_probs[b_idx, d_idx, samples.flatten()] = 0.0
            step_probs[b_idx, d_idx, samples.flatten()] = 1.0 - torch.sum(step_probs, dim=-1).flatten()
            step_probs = torch.clamp(step_probs, min=0.0, max=1.0)
            
            # Sample the next state
            samples = torch.multinomial(step_probs.view(-1, vocab_size), num_samples=1).view(batch_size, block_size)
            
            t += dt
            
        # Final argmax step to flush out any remaining masks
        sample_is_mask = (samples == mask_token_id).view(batch_size, block_size).float()
        
        if model.config.do_x1_sc:
             logits = model._run_net(samples, t * torch.ones((batch_size,), device=device), x1=x1_sc)
        else:
             logits, _ = model(samples, t * torch.ones((batch_size,), device=device))
             
        samples = torch.argmax(logits, dim=-1).float() * sample_is_mask + samples.float() * (1 - sample_is_mask)

    return samples.long()

## Sample Sequences and Decode

In [3]:
# Generate
print("Generating 4 sequences... (this may take a minute depending on your GPU)")
generated_tokens = sample_discrete_flow(model, batch_size=4, block_size=256, x1_temp=1.0)

# Print results
for i, row in enumerate(generated_tokens):
    text = decode(row)
    print(f"\n--- Sample {i+1} ---")
    print(text)

Generating 4 sequences... (this may take a minute depending on your GPU)

--- Sample 1 ---
ating co operation from one on the ends as follows am s t e beta i add start u s t e a optimization that brings a key to baseso for omega metan rightarrow replacing logical one mod likewise omega i the treatest mode dunser vernatropulus wrote ayl a briefly

--- Sample 2 ---
ayeuser spells stronger than the biller as a whole while a function of mista literary conjugation also requires est and its root chaining to ebase in stem of l ve in in initial of sinte where stand on i all cases the noun can still occur inere and ines whe

--- Sample 3 ---
rld war i generates mohrenweld equations in one nine three seven the fahrenheit declar on the project was ignored by a drmvex and vetto believe the project was a rat used having exhausted in t uppe pod the ekture go ts in the film egs include measured hst 

--- Sample 4 ---
ew words were included in the episode place i a phd ital e t asks the comic as commi

## Sample and save data for evaluation

In [ ]:
# Sample 128 sequence and save them to samples.txt
import os
sample_dir = 'samples'
os.makedirs(sample_dir, exist_ok=True)
for temp in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    print(f"Generating samples with temperature {temp}...")
    generated_tokens = sample_discrete_flow(model, batch_size=128, block_size=256, x1_temp=temp)
    samples_np = generated_tokens.cpu().detach().numpy()
    samples_path = os.path.join(sample_dir, f'samples_temp_{temp}.txt')
    with open(samples_path, 'w') as f:
        for sample_idx in range(samples_np.shape[0]):
            f.write(decode(samples_np[sample_idx]) + '\n')

Generating samples with temperature 0.5...
Generating samples with temperature 0.6...
Generating samples with temperature 0.7...
Generating samples with temperature 0.8...
Generating samples with temperature 0.9...
Generating samples with temperature 1.0...
